# Занятие 5. Файлы и исключения

**План занятия**

1. Кодировки и Юникод
2. Чтение и запись
3. Разбор записей
4. Исключения
5. Стратегия для грязных данных
6. Домашние задачи

**Теория:** `theory/05_Файлы_и_исключения.md`
Т. Гэддис, гл. 6 (с. 308 / PDF 333)

In [ ]:
# Эта ячейка находит папку с данными. Запустите её ПЕРВОЙ.
import os

CANDIDATES = ["../data", "data", "./data", "/content/data",
              "/content/drive/MyDrive/mglu/data"]
DATA = next((p for p in CANDIDATES if os.path.isdir(p)), None)
print("Data folder:", os.path.abspath(DATA) if DATA else "NOT FOUND")

---

## 1. Кодировки

Юникод присваивает номер каждому символу. UTF-8 задаёт способ записать эти номера
байтами.

In [ ]:
for ch in "aаЁ":
    print(f"{ch}  code {ord(ch):>6}  {len(ch.encode('utf-8'))} bytes")

print()
print("символов:", len("привет"), " байт:", len("привет".encode("utf-8")))

### Визуально одинаковые символы

**Проверьте себя.** Что напечатает следующая ячейка?

In [ ]:
cyrillic = "а"
latin = "a"

print(cyrillic, latin)
print(cyrillic == latin)
print(ord(cyrillic), ord(latin))

Единственная ошибка курса, которую нельзя увидеть при чтении кода.
На экране символы одинаковы, сравнение даёт ложь.

Способ проверить:

In [ ]:
suspicious = "аaа"
print([ord(c) for c in suspicious])
print("латиница внутри:", [c for c in suspicious if ord(c) < 128])

Сюда же относятся дефис, минус и тире:

In [ ]:
for ch in "-\u2212\u2013\u2014":
    print(f"{ch}  U+{ord(ch):04X}")

### Что бывает при неверной кодировке

In [ ]:
probe = "probe_encoding.txt"

with open(probe, "w", encoding="cp1251") as f:
    f.write("Привет, мир")

try:
    with open(probe, encoding="utf-8") as f:
        print("as utf-8:", f.read())
except UnicodeDecodeError as error:
    print("UnicodeDecodeError:", error)

with open(probe, encoding="cp1251") as f:
    print("as cp1251:", f.read())

os.remove(probe)

---

## 2. Чтение и запись

In [ ]:
with open(f"{DATA}/text_clean.txt", encoding="utf-8") as f:
    text = f.read()

print("символов:", len(text))
print("слов:", len(text.split()))
print()
print(text[:200], "...")

`with` закрывает файл при выходе из блока, включая выход по исключению.

Построчный обход основной: файл на гигабайт через `read()` не пройдёт.

In [ ]:
with open(f"{DATA}/dialog.txt", encoding="utf-8") as f:
    for number, line in enumerate(f, start=1):
        if number > 4:
            break
        print(f"{number}: {line.rstrip(chr(10))!r}")

Без `rstrip` каждая строка кончается переводом строки, и сравнения ломаются:

In [ ]:
raw = "word\n"
print(raw == "word")
print(raw.rstrip("\n") == "word")

| Режим | Действие |
|---|---|
| `"r"` | чтение, по умолчанию |
| `"w"` | запись, содержимое стирается |
| `"a"` | дописывание в конец |

`"w"` стирает файл сразу при открытии, ещё до всякой записи.

---

## 3. Разбор записей

In [ ]:
records = [
    "Pushkin,1799,poet",
    "Gogol,1809,writer",
    "Lermontov,about 1814,poet",
    "broken line",
]

for line in records:
    fields = line.split(",")
    print(f"{len(fields)} полей: {fields}")

Третья запись имеет три поля, но год не число. Четвёртая имеет одно поле.
Наивный разбор упадёт на обеих.

---

## 4. Исключения

Три способа обработать те же данные.

In [ ]:
print("=== 1. Без обработки ===")
try:
    for line in records:
        name, year, role = line.split(",")
        print(f"  {name}: {int(year)}")
except Exception as error:
    print(f"  упало: {type(error).__name__}: {error}")

In [ ]:
print("=== 2. except: pass ===")
parsed = []
for line in records:
    try:
        name, year, role = line.split(",")
        parsed.append((name, int(year)))
    except:
        pass                       # так делать нельзя

print(f"  обработано {len(parsed)} из {len(records)}")
print(f"  {parsed}")
print("  половина данных исчезла, и никто об этом не узнал")

In [ ]:
print("=== 3. Считаем и показываем ===")
parsed = []
skipped = []
for line in records:
    try:
        name, year, role = line.split(",")
        parsed.append((name, int(year)))
    except ValueError as error:
        skipped.append((line, str(error)))

print(f"  обработано {len(parsed)}, пропущено {len(skipped)}")
for line, reason in skipped:
    print(f"    {line!r}: {reason}")

Сравните два исхода:

* программа упала на третьей записи из ста, вы об этом знаете,
* программа обработала все сто, молча пропустив тридцать, результат занижен
  на треть, в отчёте ни слова.

Второй хуже.

In [ ]:
# ловим конкретное исключение, а не всё подряд
path = f"{DATA}/no_such_file.txt"

try:
    with open(path, encoding="utf-8") as f:
        content = f.read()
except FileNotFoundError:
    print(f"File not found: {path}")
    content = ""

print("длина:", len(content))

---

## 5. Функция очистки

Реальный текстовый файл содержит лишние пробелы, остатки разметки, разные виды
кавычек и тире.

In [ ]:
with open(f"{DATA}/text_dirty.txt", encoding="utf-8") as f:
    dirty = f.read()

print(repr(dirty[:300]))

In [ ]:
import re

def clean(text):
    """Приводит текст к нормализованному виду. Возвращает текст и журнал шагов."""
    log = {"original": len(text)}

    text = re.sub(r"<[^>]+>", " ", text)
    log["no tags"] = len(text)

    text = text.replace("&nbsp;", " ").replace("&laquo;", '"').replace("&raquo;", '"')
    log["no entities"] = len(text)

    for quote in "«»“”":
        text = text.replace(quote, '"')
    for dash in "\u2014\u2013\u2212":
        text = text.replace(dash, "-")
    log["quotes unified"] = len(text)

    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    log["spaces collapsed"] = len(text)

    return text, log


cleaned, log = clean(dirty)
for step, size in log.items():
    print(f"{step:>18}: {size}")
print()
print(cleaned[:200], "...")

### Что очистка потеряла

In [ ]:
print("заглавных слов было:", sum(1 for w in dirty.split() if w[:1].isupper()))
print("стало:", sum(1 for w in cleaned.split() if w[:1].isupper()))
print()
print("кавычек-ёлочек было:", dirty.count("«"), " стало:", cleaned.count("«"))
print()
print("тире было:", dirty.count("\u2014"), " дефисов было:", dirty.count("-"))
print("дефисов стало:", cleaned.count("-"), "различить уже нельзя")

После такой очистки нельзя искать имена собственные, отличать прямую речь
от цитаты, отличать тире от дефиса.

Для подсчёта частот это приемлемо. Для извлечения имён нет. Универсально правильной
очистки не существует, есть подходящая под задачу, и вы должны уметь перечислить,
что она теряет.

In [ ]:
# что осталось нечищеным
from collections import Counter

leftovers = Counter(c for c in cleaned if not (c.isalnum() or c.isspace()))
for ch, n in leftovers.most_common():
    print(f"  {ch!r} (U+{ord(ch):04X}): {n}")

---

# Домашние задачи

Рассчитаны примерно на 30 минут.

### Задача 1. Трассировка (без запуска)

Что напечатает код и какое число окажется в `total`?

In [ ]:
# Мой ответ: ...

# values = ["10", "20", "thirty", "40"]
# total = 0
# for v in values:
#     try:
#         total += int(v)
#     except:
#         pass
# print(total)

# Второй вопрос: как узнать, что одна запись потерялась?

### Задача 2. Минимум

Доработайте функцию `clean` под свой набор правил и **письменно перечислите,
что она теряет**. Список потерь важнее кода.

LeetCode работу с файлами и кодировками не покрывает, это единственное занятие
курса без задачи с сайта.

In [ ]:
def my_clean(text):
    # ваш код здесь
    pass

# Что теряет моя очистка:
# 1.
# 2.
# 3.

### Задача 3. Безопасное чтение

Напишите `read_safely(path)`, возвращающую текст файла. При `FileNotFoundError`,
`UnicodeDecodeError` и `PermissionError` печатайте понятное сообщение
и возвращайте пустую строку.

In [ ]:
def read_safely(path):
    # ваш код здесь
    pass

# print(len(read_safely(f"{DATA}/text_clean.txt")))
# print(len(read_safely(f"{DATA}/missing.txt")))

### Задача 4. Разбор CSV со счётчиком

Прочитайте `data/frequencies.csv` (формат `word,count`), пропустите заголовок,
битые строки считайте отдельно. Выведите топ-5 по частоте и число пропущенных.

In [ ]:
with open(f"{DATA}/frequencies.csv", encoding="utf-8") as f:
    print("заголовок:", f.readline().strip())
    print("первая запись:", f.readline().strip())

# ваш код здесь

### Задача 5. Подумать (кода не нужно)

Вам дали корпус из 500 файлов. Часть в UTF-8, часть в CP1251, какие именно
неизвестно. Опишите стратегию: как прочитать всё, ничего не потеряв молча.

### Задача 6. Трек «алгоритмы» (по желанию)

1108 Defanging an IP Address, 434 Number of Segments in a String,
2114 Maximum Number of Words Found in Sentences.

---

# Итоги

- `encoding` указывают всегда. Без него код ломается на другой машине.
- `"w"` стирает файл в момент открытия.
- Большие файлы читают построчно.
- `rstrip("\n")` обязателен при построчном чтении.
- `except: pass` превращает падающую программу в тихо врущую.
- Пропущенные записи считают и показывают.
- Визуально одинаковые строки бывают разными. Сравнивайте коды символов.
- Очистка текста это набор решений, и каждое имеет цену.